# See-through su Google Colab (T4 / A100)

Esegue **see-through** (decomposizione di illustrazioni anime in layer 2.5D + PSD) senza GPU locale, direttamente su Colab.

**Prima di eseguire** (una volta sola): `Runtime > Ambiente di esecuzione > Change runtime type` → scegli la GPU:
- **T4 GPU** (gratis, 16 GB VRAM) — raccomandata
- *A100 high-RAM* (a pagamento) — più veloce, qualità 1280

Note Colab free: sessione ~12h e disco non stabile → **collega Google Drive** (cella 4) per non perdere input e PSD.
Esegui le celle in ordine col pulsante ▶.

## 1) Verifica GPU + torch

In [ ]:
!nvidia-smi
import torch
ok = torch.cuda.is_available()
print('CUDA:', ok, '| GPU:', torch.cuda.get_device_name(0) if ok else 'NESSUNA - serve una GPU (T4) nel runtime')
if ok:
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print('torch:', torch.__version__)

## 2) Clona il repo + installa dipendenze MINIMALI

Installa solo quanto serve all'inferenza (niente Qt/training) e imposta le versioni HF che il codice vuole (`diffusers==0.37.0`, `transformers==5.0.0`). Il primo giro richiede un paio di minuti.

In [ ]:
import os, subprocess, pathlib
os.chdir('/content')

if not os.path.exists('/content/see-through'):
    subprocess.run(['git','clone','--depth','1','https://github.com/paolo1234/see-through.git'], check=True)
    os.makedirs('/content/see-through/assets', exist_ok=True)   # symlink assets o cartella vuota

pips = ['pip','install','-q','--upgrade',
        '-e','/content/see-through/common',
        '-e','/content/see-through/annotators',
        'diffusers==0.37.0','transformers==5.0.0','accelerate','huggingface-hub','safetensors',
        'psd-tools[composite]','opencv-python-headless','scikit-learn','scipy','scikit-image',
        'einops','click','tqdm','kornia','omegaconf','grad-cam','pyyaml','pillow']
subprocess.run(pips, check=True)
print('deps OK')

## 3) Prepara i percorsi di import

il progetto usa `from utils...` / `from modules...`: va aggiunto il path di `common/` e `annotators/`.

In [ ]:
import os, sys
sys.path.insert(0, '/content/see-through')
sys.path.insert(0, '/content/see-through/common')
sys.path.insert(0, '/content/see-through/annotators')
os.chdir('/content/see-through')
print('CWD:', os.getcwd())

## 4) Monta Drive + cartelle di lavoro + CACHE MODELLI su Drive
Oltre a input/output spostiamo la **cache HuggingFace su Drive**:
(~7 GB) così i modelli si scaricano **solo il primo giro** e sono poi riusati,
anche nelle sessioni successive (il check che esistano è automatico di HF).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path('/content/drive/MyDrive/see-through-work')
(WORK/'input').mkdir(parents=True, exist_ok=True)
(WORK/'output').mkdir(parents=True, exist_ok=True)
(WORK/'inputs').mkdir(parents=True, exist_ok=True)
(WORK/'hf_home').mkdir(parents=True, exist_ok=True)
print('WORK:', WORK)

# reindirizza la cache HuggingFace su Drive -> modelli persistenti
import os
os.environ['HF_HOME']       = str(WORK/'hf_home')
os.environ['HF_HUB_CACHE']  = str(WORK/'hf_home'/'hub')
os.environ['XDG_CACHE_HOME']= str(WORK/'hf_home'/'xdg')
os.makedirs(os.environ['HF_HUB_CACHE'], exist_ok=True)
print('Cache HF su Drive:', os.environ['HF_HOME'])

# controllo: i modelli sono gia scaricati?
hub = pathlib.Path(os.environ['HF_HUB_CACHE'])
models = list(hub.glob('models--*')) if hub.is_dir() else []
if models:
    names = [m.name.replace('models--','').replace('--','/') for m in models]
    gb = sum(f.stat().st_size for m in models for f in m.rglob('*') if f.is_file())/1e9
    print('Modelli GIA in cache su Drive:', names)
    print(f'Dimensione cache: {gb:.1f} GB -> nessun re-download')
else:
    print('Cache vuota: al primo avvio i modelli andranno su Drive (~7 GB).')
    print('Le volte successive non li riscarichera.')

## 5) Login HuggingFace (opzionale)

Se un modello chiede autenticazione, incolla qui il tuo `token` (huggingface.co/settings/tokens). Altrimenti salta.

In [ ]:
from huggingface_hub import login
TOKEN = ''   # incolla qui il tuo token HF (o lascia vuoto per saltare)
if TOKEN:
    login(token=TOKEN)
    print('login OK')
else:
    print('token vuoto -> login saltato, proviamo a scaricare pubblicamente')

## 6) Carica la tua immagine

Una illustrazione anime con sfondo il più possibile pulito/mono. La qualità dei layer dipende molto dalla nitidezza del soggetto.

In [ ]:
from google.colab import files

print('Seleziona la tua immagine (PNG/JPEG)...')
up = files.upload()
IMGDIR = WORK/'input'
for fn, content in up.items():
    dst = IMGDIR / fn
    dst.write_bytes(content if isinstance(content, bytes) else content.getvalue())
    print('Salvata:', dst)

## 7) Lancia l'inferenza

Tuned per **T4 (16 GB)**: risoluzione diffusion **1024**, depth **768**, `--group_offload` automatico se VRAM < 17 GB, output `.psd`.

Il primo avvio scarica ~6 GB di modelli (1-2 min), poi elabora in 1-3 min.

In [ ]:
import sys as _sys, subprocess
IMGDIR = WORK/'input'
imgs = [p for p in sorted(IMGDIR.rglob('*')) if p.suffix.lower() in ('.png','.jpg','.jpeg','.webp')]
assert imgs, 'Nessuna immagine trovata (cella 6)'
SRC = str(imgs[0])
print('Immagine:', SRC)

LAYER_RES = 1024     # T4 stabile; 1280 solo su A100
DEPTH_RES = 768      # risoluzione depth
STEPS     = 30       # step diffusion layer
OUTDIR    = WORK/'output'

vr = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0
cmd = [_sys.executable, 'inference/scripts/inference_psd.py',
       '--srcp', SRC,
       '--save_dir', str(OUTDIR),
       '--resolution', str(LAYER_RES),
       '--resolution_depth', str(DEPTH_RES),
       '--inference_steps', str(STEPS),
       '--inference_steps_depth', '4',
       '--save_to_psd']
if vr < 17:
    cmd.append('--group_offload')

print('CMD:', ' '.join(cmd))
print('=== Esecuzione... ===')
r = subprocess.run(cmd)
print('Exit code:', r.returncode)

## 8) Visualizza i layer (check qualitativo)

Mostra ogni layer RGBA su sfondo check: è il modo rapido per valutare separazione e allucinazioni prima di aprire il PSD.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

OUTDIR = WORK/'output'
srcname = pathlib.Path(SRC).stem
tgt = OUTDIR / srcname
pngs = sorted(tgt.glob('*.png')) if tgt.exists() else []
print('Layer PNG:', [p.name for p in pngs])

def checker_bg(h, w, s=8):
    base = Image.new('RGB', (w, h), (255,255,255))
    d = ImageDraw.Draw(base)
    for y in range(0, h, s):
        for x in range(0, w, s):
            if (x//s + y//s) % 2 == 0:
                d.rectangle([x, y, x+s-1, y+s-1], fill=(200,200,200))
    return base

cols = 4
for start in range(0, len(pngs), cols):
    row = pngs[start:start+cols]
    imgs = []
    for p in row:
        im = Image.open(p).convert('RGBA')
        im.thumbnail((256,256))
        bg = checker_bg(*im.size).convert('RGBA')
        bg.alpha_composite(im)
        imgs.append(bg)
    W = sum(i.size[0]+8 for i in imgs) - 8; H = max(i.size[1] for i in imgs)
    canvas = Image.new('RGB', (W, H), (255,255,255))
    x = 0
    for im in imgs:
        canvas.paste(im, (x, 0)); x += im.size[0]+8
    plt.figure(figsize=(14, 4)); plt.imshow(canvas); plt.axis('off'); plt.show()

print('\nPSD salvati in:')
for psp in sorted(OUTDIR.rglob('*.psd')):
    print(' -', psp)

## 9) (Opzionale) Scarica il PSD

Scarica sul tuo PC i file PSD generati.

In [ ]:
from google.colab import files as gfiles
srcname = pathlib.Path(SRC).stem
for psp in sorted(OUTDIR.rglob('*.psd')):
    if psp.stem.replace('_depth','') == srcname:
        gfiles.download(str(psp))
print('download avviati se presenti')


## 10) GUI Studio (opzionale, Gradio)
Carica il file `see_through_studio.py` (fornito) in `/content/see-through` o nella cartella corrente, poi esegui le due celle sotto.
La UI fa: esegui pipeline per stadi, mostra layer/depth, **editor del singolo layer**, scelta+ordine e ricomposizione, export PSD.


In [ ]:
!pip install -q gradio
print('gradio OK')

In [ ]:
# CARICA il file see_through_studio.py accanto a questa cartella/notebook
import sys, pathlib, shutil
# se il file non c'è, lo può copiare da qui (ad es. da Drive o put in /content)
src = pathlib.Path('/content/see_through_studio.py')
if src.exists():
    shutil.copy(src, '/content/see-through/see_through_studio.py')
sys.path.insert(0, '/content/see-through')
import see_through_studio as st
st.app.queue().launch(share=True)   # apre un link pubblico temporaneo
